# Variational Classifier — Iris Dataset


## Librerias

In [42]:
# 1. Import all libraries
import pandas as pd
import time
import matplotlib.pyplot as plt
import pennylane as qml
from sklearn import datasets
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from pennylane import numpy as np
from pennylane.optimize import NesterovMomentumOptimizer

np.random.seed(42)


## Data

In [ ]:
data = pd.DataFrame(datasets.load_iris().data, columns=datasets.load_iris().feature_names)
data['target'] = datasets.load_iris().target
data

In [ ]:
data.target.unique()

## Data Preparation

In [ ]:
n_qubits = 4

X = data.drop(columns='target')
y = data['target']

# Split test set with a ratio of 0.2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Custom Dataset Loaded: Train={len(X_train)}, Test={len(X_test)}")

# Normalize to [0, π]
minmax = MinMaxScaler(feature_range=(0, np.pi))
X_train = minmax.fit_transform(X_train)
X_test  = minmax.transform(X_test)
X_test  = np.clip(X_test, 0, np.pi)

print(f"Final shapes -> Train: {X_train.shape}, Test: {X_test.shape}")

## Variational Classifier

In [ ]:
dev = qml.device("default.qubit")

### Variational Data Prep

In [ ]:
X = np.array(X, dtype=float)
y = np.array(y)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X = scaler.fit_transform(X)

In [ ]:
def layer(layer_weights):
    for wire in range(4):
        qml.Rot(*layer_weights[wire], wires=wire)
    for wires in ([0, 1], [1, 2], [2, 3], [3, 0]):
        qml.CNOT(wires=wires)

In [ ]:
def state_preparation(x):
    for i in range(4):
        qml.RY(x[i], wires=i)

In [ ]:
@qml.qnode(dev)
def circuit(weights, x):
    state_preparation(x)
    for layer_weights in weights:
        layer(layer_weights)
    return [
        qml.expval(qml.PauliZ(0)),
        qml.expval(qml.PauliZ(1)),
        qml.expval(qml.PauliZ(2)),
    ]

def variational_classifier(weights, bias, x):
    out = qml.math.stack(circuit(weights, x))
    return out + bias

def cost(weights, bias, X, Y):
    predictions = qml.math.stack(
        [variational_classifier(weights, bias, x) for x in X]
    )
    Y = np.array(Y, dtype=int)
    targets = np.eye(3)[Y]
    return qml.math.mean((predictions - targets) ** 2)

In [ ]:
import time

# --- Tiempo de entrenamiento ---
np.random.seed(0)
weights = 0.01 * np.random.randn(2, 4, 3, requires_grad=True)
bias = np.zeros(3, requires_grad=True)
opt = NesterovMomentumOptimizer(0.1)
batch_size = 5

start = time.time()
for it in range(100):
    batch_index = np.random.randint(0, len(X), (batch_size,))
    X_batch = X[batch_index]
    Y_batch = y[batch_index]
    weights, bias = opt.step(cost, weights, bias, X=X_batch, Y=Y_batch)

    predictions = [np.argmax(variational_classifier(weights, bias, x)) for x in X]
    current_cost = cost(weights, bias, X, y)
    acc = np.mean(np.array(predictions) == np.array(y))
    print(f"Iter: {it+1:4d} | Cost: {current_cost:0.7f} | Accuracy: {acc:0.7f}")
train_time = time.time() - start
print(f"\nTraining time: {train_time:.4f} s")

In [ ]:
# Predicción
def predict(weights, bias, X):
    return np.array([
        np.argmax(variational_classifier(weights, bias, x))
        for x in X
    ])

        # --- Tiempo de predicción (inferencia) ---
start = time.time()
y_test_pred = predict(weights, bias, X_test)
infer_time = time.time() - start

acc_test = float(np.mean(np.array(y_test_pred) == np.array(y_test)))
print("Test accuracy:", acc_test)
print(f"Inference time: {infer_time:.4f} s")
print(f"Training time:  {train_time:.4f} s")

In [ ]:
y_test_pred = predict(weights, bias, X_test)
acc_test = float(np.mean(np.array(y_test_pred) == np.array(y_test)))
print("Test accuracy:", acc_test)